# 02 Spark Processing and Analysis

This notebook reads the cleaned dataset produced by notebook 01, performs transformations and aggregations using PySpark, and saves the results for the MongoDB loading step.

**Input:** `../data/processed/orders_core.parquet`  
**Output:** `../data/processed/customer_spending.parquet`, `top_categories.parquet`, `monthly_trends.parquet`, `customer_segments.parquet`

## 1. Environment Setup

Set JAVA_HOME so PySpark can locate the Java runtime before starting the Spark session.


In [ ]:
import os
import sys
import shutil
# Use the Java binary from the currently active environment
java_bin = shutil.which("java")

if java_bin is None:
    raise RuntimeError("Java not found in PATH. Activate the correct conda environment that has Java installed.")

# Set JAVA_HOME dynamically based on the resolved java path
os.environ["JAVA_HOME"] = os.path.dirname(os.path.dirname(java_bin))
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print("Python:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("java:", java_bin)

## 2. Start Spark Session

Initialize the Spark session used for all processing in this notebook.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Olist_Spark_Processing").getOrCreate()
print("Spark version:", spark.version)

## 3. Load Processed Dataset

Read the cleaned and joined dataset produced by the ingestion notebook.

In [ ]:
df = spark.read.parquet("../data/processed/orders_core.parquet")
df.printSchema()
print("Total rows:", df.count())
df.cache()

## 4. Total Spending Per Customer

Calculate how much each unique customer has spent across all their orders. Payment values are deduplicated at order level before aggregating to avoid double counting from the order-item level structure.

In [ ]:
from pyspark.sql.functions import col, sum, count, round

# deduplicate payment_value_total per order first to avoid double counting
from pyspark.sql.functions import first

spending_per_customer = (
    df.dropna(subset=["price"])
    .groupBy("customer_unique_id", "order_id")
    .agg(first("payment_value_total").alias("order_payment"))
    .groupBy("customer_unique_id")
    .agg(
        round(sum("order_payment"), 2).alias("total_spent"),
        count("order_id").alias("total_orders")
    )
    .orderBy(col("total_spent").desc())
)

spending_per_customer.show(10)
print("Unique customers:", spending_per_customer.count())
spending_per_customer.cache()

## 5. Top Product Categories by Revenue

Identify which product categories generate the most revenue and have the highest sales volume.

In [ ]:
top_categories = (
    df.dropna(subset=["price"])
    .groupBy("product_category_name")
    .agg(
        round(sum("price"), 2).alias("total_revenue"),
        count("order_item_id").alias("total_items_sold")
    )
    .orderBy(col("total_revenue").desc())
)

top_categories.show(15)

## 6. Monthly Order Trends

Analyse how order volume and revenue change month by month across 2016 to 2018.

In [ ]:
from pyspark.sql.functions import concat, lit, lpad

monthly_trends = (
    df.dropna(subset=["price"])
    .select("order_id", "year", "month", "payment_value_total")
    .dropDuplicates(["order_id"])
    .groupBy("year", "month")
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("payment_value_total"), 2).alias("monthly_revenue")
    )
    .orderBy("year", "month")
)

monthly_trends.show(30)

## 7. Customer Segmentation

Segment customers into High, Medium, and Low spenders based on their total spending to understand the distribution of customer value.

In [ ]:
from pyspark.sql.functions import when

segmented = spending_per_customer.withColumn(
    "segment",
    when(col("total_spent") >= 500, "High Spender")
    .when(col("total_spent") >= 100, "Medium Spender")
    .otherwise("Low Spender")
)

segment_summary = (
    segmented.groupBy("segment")
    .agg(
        count("customer_unique_id").alias("customer_count"),
        round(sum("total_spent"), 2).alias("segment_revenue")
    )
    .orderBy(col("segment_revenue").desc())
)

segment_summary.show()

## 8. Spark SQL Queries

Register DataFrames as temporary views and run SQL queries to analyse customer behaviour, category performance and monthly trends.

In [ ]:
# Register views for Spark SQL
spending_per_customer.createOrReplaceTempView("customer_spending")
top_categories.createOrReplaceTempView("category_revenue")
monthly_trends.createOrReplaceTempView("monthly_trends")
segmented.createOrReplaceTempView("customer_segments")

# Top 10 customers by spending
print("=== Top 10 Customers by Total Spending ===")
spark.sql("""
    SELECT customer_unique_id, total_spent, total_orders
    FROM customer_spending
    ORDER BY total_spent DESC
    LIMIT 10
""").show()

# Top 5 categories by revenue
print("=== Top 5 Product Categories by Revenue ===")
spark.sql("""
    SELECT product_category_name, total_revenue, total_items_sold
    FROM category_revenue
    ORDER BY total_revenue DESC
    LIMIT 5
""").show()

# Best performing month
print("=== Best Performing Month by Revenue ===")
spark.sql("""
    SELECT year, month, total_orders, monthly_revenue
    FROM monthly_trends
    ORDER BY monthly_revenue DESC
    LIMIT 5
""").show()

# Customer segment breakdown
print("=== Customer Segment Summary ===")
spark.sql("""
    SELECT segment, COUNT(*) as customer_count,
    ROUND(SUM(total_spent), 2) as total_revenue
    FROM customer_segments
    GROUP BY segment
    ORDER BY total_revenue DESC
""").show()

## 9. Save Results for MongoDB Loading

Write all aggregated results to parquet files in the processed folder for next notebook to load into MongoDB.

In [ ]:
spending_per_customer.write.mode("overwrite").parquet("../data/processed/customer_spending.parquet")
top_categories.write.mode("overwrite").parquet("../data/processed/top_categories.parquet")
monthly_trends.write.mode("overwrite").parquet("../data/processed/monthly_trends.parquet")
segmented.write.mode("overwrite").parquet("../data/processed/customer_segments.parquet")

print("All outputs saved successfully")